# Trabajo final · Data Scientist · Cobijo

**Portal inmobiliario** · Empresa ficticia. Los datos son sintéticos y **se generan por código en este notebook**: ningún resultado representa una situación real.

## La pregunta oficial

> **¿Cuánto vale una propiedad, y con qué margen de error podemos mostrarlo?**

**Decisión que habilita:** El precio sugerido, el rango que se muestra y a qué propiedades no se les da tasación automática.

**Tipo de problema:** Regresión tabular con intervalos de error.

Las instrucciones completas (qué se entrega, criterios de calidad y defensa) están en el PDF **«Trabajo final · Data Scientist · Instrucciones»**. Este notebook es tu lugar de trabajo: vas de los datos sucios a la recomendación sin salir de acá.

| Sección | Módulo del programa |
| --- | --- |
| 1 | Introducción: definición del problema |
| 2 | Python: limpieza |
| 3 | Estadística |
| 4 | Machine Learning y regresión |
| 5 | Deep Learning |
| 6 | Apache Superset |
| 7 | De modelo a producción |
| 8 | Ciencia de datos aplicada al negocio |
| 9 | Fichas y recomendación |

## 0 · Preparación

Ejecutá esta celda una sola vez. `xgboost` es opcional: si querés usarlo, descomentá la última línea.

In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib
# %pip install -q xgboost


## 1 · Datos y definición del problema

La celda siguiente **genera las tablas**. Tarda unos segundos. Después mirá qué hay con `inventario()` y `diccionario()`.

Antes de modelar, definí el problema: **tipo de problema, población, período, granularidad y variable objetivo**, y qué información está realmente disponible en el momento de decidir.

In [ ]:
import zlib, base64
TEXTO = (
    "eNqVWUtzG8cRvutXTK2t1K4FrBYgIVGUmAolx1VJybJi2a5yIQhqgB0SK+3L+wBJqXjIT8hP8NEHHxzfUjmFfyxfd8++QFBOVCIw"
    "29PT3dPvXkRJnhWVSuskv1K6VGk+UrlOQyzxPw/v3QvNmTo3qSl0sQx1lZVuaZIojvXJNJg+ejRSW1OEOjz5Qsel8Y7vKfwr1Ako"
    "+QUIZYkPCrqOq2WRnjdnvafq1XMgTabBU/CL9aoAuxM1d07rIiu0M1LOCxMWGS2+jAodZoXAohXDXupcrze8/EttcNo+3Pw91gmv"
    "Xmmwf8vLN7peG1r8Wa8JGt78SE+n8bkuikwIvtjom180M/kG8vDitdnUNz+lvJ1Fm5pp3PwjKnjxpY4qu/tS14WJafVVHG2FXpQX"
    "Nz+VTC+KGfRVsrr5V8OAvv9Y5lEKdWTOgtWWF+YsepuJIr4jPRHWcwgZyaWzBFoKmftrXfwgAp3GMIqFvsjKSstNi/DmZxbudW2K"
    "CiyeqjRLVoVh8mfOh/xafbi4dtRZVqhcRWnHniAXDLGWwdnCLzf12VlsXEvF69Ozq/nxq+dylUsAC79OI9BK3GCkDvD36jkOXd25"
    "E0ZlJY6zucqzyr1UYzWZjXCCvoHwPku1IEAl+sqFt7CC+Ob08TlrlT6+cBbeHIjrOMpdl0+sK51O3StLVIh76gGRyyNPPXyoXF6q"
    "h+rAGylINfN8XVZXuXGjtPLkYuZsacX4AO7HKvCJGmQ4VuPAD2YsCoGnLA9DpywVAXn/C4FOrp+q0hTbaB2xyRtpjwKINvWn6jPR"
    "yAOoK4WudEzaOmJlQbaRmgSBxzLF2VrHQuG9KRChwOANMuSSDAmnPzfugXcM4Ce4g1lXmT1W1nobvUeAKWQCBPM6QvSHfJxDuY6w"
    "dTKQAbIfWZtdCFtzmbvjznDzY2xnqVngJpdzWo3U8QKW7G9ctRtQvjr0A5ATkR6cKPdC/UF40+aFX9aJqy+j8mTiNWZYcVwM/MEa"
    "Z/5+wVd/T1cnADiMSfuTnlLxGBzi0e1sMFazgDyCpWA2iS7e1fly1TMPLh/I8Rmd7gQZd0L5idGp65HoHaysQtfbsSaoTK1BsTyk"
    "z6lcUA6RX+Sh/7mu9BeFToz7wZENuJCNOTgUXREA+sIT3U+n60gv1yatimz5LsEmQf0iq9PQnYCbAytHa7NsLw+Udm3xAsK7lOOX"
    "FjYl2JXArlrYtcj86jXEnR4FAZL6in0GWRlc4I+45GtUlpPGVwJ/1pjiociW6EvR2G+hsC94cJYqyrM+l7kTmlwXFfSUVpww17rk"
    "bJhvnEUjwdx/BOf1p1C2HxwtRPJk2rOwFaC1E54vNqYwrjA8aeke+o8Rhre3wY42D+gjmHls3AOKmNe0nsLijwLydp2shmxFnxDm"
    "oZoe3nKVxw2FCdLAIDVB3TodZpGWGjF5yPlkl96sR2+Y6lgnOq2i8x7Fwj/XSaLdqY/Tk0M+zGnySeBZXhBkna03bBQp/i6Q1LNe"
    "8Ext6E3IuIk88XpHu0I68J+wnkzKNPMsKsssdSecHbsTA7t7TFPkkyy4RJWVQPomSgyiI8ldB93LwTiY4L8DFgmXscLH7c25KUou"
    "TAdMA45m0lAUIYYm5GdoXo5GbRZJKBCbh8mRTTcHJCVho9YceTZZb0HqsX90wGIePQIKCMfZOexOorcJY75aiHYIhdXKmntMNK25"
    "QdZmtuAR6VBM1iYn1htvHw3UBf8UJU326B0bfN9db5lMO41udZwVXeaPt6RBTpU9f2lyp9xjJ+vNrPuMJQMiPqjAx5S6QELoQ7YJ"
    "TgodYRxGugyAwEr9DK0DUx4YLRCvZvRPVJajcUVOS4G/NqgS6Gyl11pu4S3EbILwYEZN4nHHE59q8BO2nlyLtDgJvJ67Duy0J/d7"
    "lKLAI4xCfTsgGvafkQ8cIWwiCGd2r76jNPFH7pR0uUxMsdZhtjdCodIDUXFD6tHuZSjwkVEOkYluhf4nCj1BmZXHSsfndUqTQJHl"
    "kUFjAD2WRuX1Ko7WOlXJzY/Ueyog4bLvpfs3jRtk6VrDmfDnzrlKcxvChu9qA7LyY5SMEZ2L9drYSWJBM8IpCMUmdbHl9XtQerax"
    "zZICTaL8AUV5lS0rBHpo4kq7jXLYdeY4uBBTSyj3pDoluwScdoYu9Zhc6tSTjISmi+oi+tfqBP0dpDKXVUEW3qH1+xNl2zM0E7EZ"
    "FqqXCF6eZXhAON366mW0QpuuQ5lXUvUlEppt4F9C/6dreLFm7OcmPgcbLnBfR1t49DbSMidVMpm80UUSNTXwtQH38wLKkRGiVDSP"
    "lFIQKbnGVV3oYfKbkDkOjwKJJWqN0fMgrwVSGNg3bncmUSjGoN6kVcWEdAeNc9OBvnOjl9Z5KCqBai2I3a6zwelSWrpm2Jiv2HQQ"
    "2qF8tWz9EegE4F1sljUC/gxdjFmu61VEKgVGMuX9pkoN8aqs0jGQXIslCbkr6QLslf1ueplQjT/wxT+GG+QqPBqQ53g9zqjHZBtD"
    "HRfWjeCS0QGTRQPmjF7TRZcNAoOafaoLyHAA00pu2Ytm7oGAz+dRDaIqsoxN2pAIMc+urTFoLmR3necLjIfiG7TujYn2GKZFHGqM"
    "GWbLuiRjuJzDG0Xuy2LToU7G1D4+5QSzjKjM4qhNQhuQyooIEwEyjPCT5E0dxnzRjjjvuhEHlI/b2YXSkDirv09aH7m2xtD6DuPt"
    "GWcQRubnja/zHMnbdd+R847UGXpHSr3DRgRNmdeyI1HedqKkmLaYrCSlb9qMFOqr8gTWcfsRh37lcMY1g6QW1eSiwXF/XrblMnhM"
    "+Qi6u1NUUZSM1INA3YzgK3GdpOXJvIvZJjqdzrRk0MWeQkqTgtpG6bqONdKHqFgqAR0k5yaDYbONUiZSx1aWN6aAGw7zpeef49L5"
    "6sq1nuBJuy81tN992ZrqzQPYKcs/nojA0y8Mhh1zyQc9a3TRlE1IUny7XDS/69RiT33p1+I5IS961aFTpvCwQSIPgt2FAcciiJm0"
    "ocdz2y55O2zJzfE5x5/fvwj1prd63dk4eDI+CBwPDE1pqqXcLoSyT74patPYeTweq7Jed1bTXRCts/yKLFIP+plTQMo1Yfl7sq/V"
    "HFCqXRROvO1+dWlf/pgkr65AdqRCymMn2eqtWVfdC43ojmj/AcfreQSnoA67XPMSVPFNUeh82B77wdk1ioGjojOgP+MXGcqg21Bu"
    "u59UZW//YLA9Obt2fNumuI5PsTJCx9xiH7bYuKavGh0oS7pH93C2y1cl//lnD2E2FYR236GOUiwy31vm6Ja47tOdfSlvi34E9W3H"
    "zKYyxtq3KGB52amKVH5JKi+rhbcXbQenaXpQ56gvm6CZ8Oktr9j/I4kYMwTSS6g/KioNXc7p129I99++4QYszz/qOPKSqk/oN10J"
    "VhBR2HdOhOExGDW+9Kn6kLNjfWYvejxiTXTOMSIBfaerDiYG1Qs684x7+Y4cinhaIuXowrkD+3DWZ/7tp28s+/+X6+MBmTefN2SY"
    "Su9MafqIt5DgYLtmZAfLc3Y+Ud3/4HLy/k/QqRNlB6GM5DYFan+XYd+QbRnfTl3iQk3v05o+uZWqIKBtNAcCJmKWI/Zw5FKUXFTV"
    "wl1tG5plVXjeaPcEFe7mhA7DHXxYQzl7DvXYNLa7dbD9AeA7n0istm38+3G2nt+OjGBfv2gvmer06Z0HZ71GssOXWU4XWbmjQ9FU"
    "MGsJEs6gs10Mph3INZ2hKjJe+/YOdrg9DnzcZyCn9m8d8sPKh8rOqCS7zv3w4f3k4f3vSWW/jfz9+H4yvg+Psq8XluKCTUd8p1ft"
    "dauukR56lrX5tLN5nF3AtcLlna4lR6a3vWR4RmZHGSHR3If82ixcNinYFNl+0x02pmOU3+FBgtCPyijFeEqJ1b6o93h7NwxJh+jP"
    "qvXGLZy//TV88Gm/1xlmhdun0UU5gaSSDbXy1KW2zcVmYx2DD282YsO77IYOcCptUEsgm86HfR3RyaaDFukun6HTwyChtw/Z1PNa"
    "G3bd5U6o8I+j/LuQtEs0ZgG9HWKWdoIBfIMe3Ol11Mvm1RT2smk7Css4yKtr+2YF81gqnO7d+/xPL14M+Q1aX6wRjdEZq7xAR247"
    "deeOQdwJb37WPGk14Jtf+f1DO5c3q31DOENkTmuAd47jPXDbIzn7h/I+qsB2RujeQ3+KxurmF3m7smeEFtjNvwmoTKpa3G6cdspI"
    "YVhOjWpgd83SLZRu323sTNaOvAW6su9bnH3RctzAVAcDnq2nx82KTSmIzvWoLdt7PW3oETujXmfzlt5w+usEgpLCm19jTT/3bqNz"
    "nvEwRig6PxDiDq/eIwcUm6LXUD/UhgZKQr751bk9lHUyWtAdQ1UjKqxg56Ce0M7e4YpJl4p+JpN5tef5MFXHsn/DLjZ7P8Y58qKK"
    "xeyixP4wJ993/jTXgWmalh1n/090zs3PDKRLdvDuZzo4cFYg7KF1mKcy44w+SRHY7n6566OlWQG8si4s1vX1vf8CNnlt5g=="
)
exec(zlib.decompress(base64.b64decode(TEXTO)).decode())

datos = generar_datos()          # genera las tablas, con una semilla fija: todos los equipos de esta empresa ven lo mismo
globals().update(datos)          # cada tabla queda disponible como variable


In [ ]:
SLUG = "06_Cobijo"
import os, sqlite3
import pandas as pd

def inventario():
    """Tablas disponibles: filas, columnas y porcentaje de celdas faltantes."""
    return pd.DataFrame([(n, len(df), len(df.columns), round(float(df.isna().mean().mean() * 100), 1)) for n, df in datos.items()], columns=["tabla", "filas", "columnas", "% de faltantes"])

def diccionario(tabla=None):
    """Qué significa cada columna. Es una descripción neutra: no dice qué está limpio ni cuándo se conoce cada dato."""
    return pd.DataFrame([(t, c, d) for t, cols in DICC.items() if tabla in (None, t) for c, d in cols.items()], columns=["tabla", "columna", "qué es"])

bitacora_calidad = pd.DataFrame(columns=["problema", "dónde", "filas afectadas", "qué decidimos y por qué"])
def anotar(problema, donde, filas, decision):
    """Deja anotado un problema de calidad y la decisión tomada. Es parte de la trazabilidad."""
    bitacora_calidad.loc[len(bitacora_calidad)] = [problema, donde, filas, decision]; return bitacora_calidad

def exportar_superset(carpeta="superset", **tablas):
    """Guarda las tablas que pases como archivo SQLite y como CSV, para conectar Apache Superset."""
    os.makedirs(carpeta, exist_ok=True); con = sqlite3.connect(f"{carpeta}/{SLUG}.db")
    for nombre, df in tablas.items():
        df.to_csv(f"{carpeta}/{nombre}.csv", index=False); df.astype({c: "object" for c in df.columns if str(df[c].dtype).startswith("str")}).to_sql(nombre, con, index=False, if_exists="replace")
    con.close(); print(f"Listo: {carpeta}/{SLUG}.db y un CSV por tabla. Conectá Superset a ese archivo."); return carpeta


In [ ]:
inventario()

In [ ]:
diccionario()

### Tu definición del problema

Completá este diccionario. Si cambia más adelante, cambiá también la respuesta y anotá por qué.

In [ ]:
definicion = {
    "tipo de problema": "",
    "población": "",
    "período": "",
    "granularidad (una fila es…)": "",
    "variable objetivo": "",
    "qué se sabe en el momento de decidir": "",
}
definicion


## 2 · Python

**Qué tenés que resolver en tu caso:** Pasar las superficies escritas como texto a números, unificar USD y pesos y las grafías de los barrios, y quitar avisos republicados y precios con un cero de más.

Limpiá **sin borrar en silencio**: cada decisión se anota con `anotar(...)` y queda en la bitácora de calidad.

In [ ]:
# Tu trabajo


In [ ]:
bitacora_calidad

## 3 · Estadística

**Qué tenés que resolver en tu caso:** Distribución del precio (asimétrica) y su logaritmo; relación con superficie y barrio, con intervalos.

Mostrá distribuciones e intervalos, y al menos una comparación o prueba con su conclusión y su límite.

In [ ]:
# Tu trabajo


## 4 · Modelado

**Qué tenés que resolver en tu caso:** Regresión lineal regularizada, Random Forest y XGBoost sobre el logaritmo del precio; validación dejando barrios o períodos afuera.

Empezá por una referencia simple. Validá sin mirar el futuro y elegí la métrica por el costo del error, no por costumbre.

In [ ]:
# Tu trabajo


## 5 · Deep Learning

**Qué tenés que resolver en tu caso:** Una red tabular con representación del barrio contra el boosting: ¿mejora lo bastante?

Entrená una red y compará contra la mejor referencia con la misma validación. Decidí si vale su costo y su opacidad.

In [ ]:
# Tu trabajo


## 6 · Superset

**Qué tenés que resolver en tu caso:** Tablero con el error por barrio y por tipo de propiedad.

Con `exportar_superset(nombre=tabla, ...)` llevás tus tablas a un archivo SQLite para conectar Apache Superset. Un gráfico por pregunta.

In [ ]:
# Tu trabajo


## 7 · Producción

**Qué tenés que resolver en tu caso:** Contrato de entrada, versión y monitoreo del error cuando cambia el mercado.

Definí el contrato de entrada (campos, tipos y rangos), la versión, una prueba de humo y las señales de monitoreo con umbral, acción y responsable.

In [ ]:
# Tu trabajo


## 8 · Negocio

**Qué tenés que resolver en tu caso:** Cuándo no dar tasación por baja confianza, y cómo explicarle el precio a quien vende, sin discriminar.

Decisión, costos, explicación global y local, y una recomendación proporcional a la evidencia.

In [ ]:
# Tu trabajo


## 9 · Fichas y recomendación

### Ficha de evidencia

| Campo | Tu respuesta |
| --- | --- |
| Decisión que se toma | |
| Población y período | |
| Procedimiento | |
| Resultado | |
| Validación | |
| Límite | |

### Ficha de modelo y plan de monitoreo

| Campo | Tu respuesta |
| --- | --- |
| Qué modelo se propone y por qué ese | |
| Con qué datos y qué versión | |
| Contrato de entrada | |
| Riesgos principales | |
| Señales a vigilar, umbral, acción y responsable | |

### Recomendación (una página)

- **Evidencia:** qué muestran los datos.
- **Interpretación:** qué creemos que significa, sin presentarlo como causalidad.
- **Recomendación:** qué conviene hacer, proporcional a la evidencia.
- **Una limitación y un siguiente paso.**